<a href="https://colab.research.google.com/github/akaleniuszka/03MIAR--Algoritmo-de-Optimizacion/blob/main/Proyecto/Proyecto_Alfredo_Kaleniuszka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de Optimización - Trabajo Práctico

Nombre y Apellidos: Alfredo Kaleniuszka  
URL GitHub: [Proyecto_Alfredo_Kaleniuszka.ipynb](https://github.com/akaleniuszka/03MIAR--Algoritmo-de-Optimizacion/blob/70b3f736c5ecbc583d9e593997840bc33cde8b9b/Proyecto/Proyecto_Alfredo_Kaleniuszka.ipynb)  

## Problema elegido

> **2. Organizar los horarios de partidos de La Liga**

## Descripción del problema

La Liga desea organizar los horarios de los partidos de una jornada de forma que se maximice la audiencia total esperada.

El problema consiste en asignar los 10 partidos de una jornada a los horarios disponibles. La audiencia de cada partido depende de:

- La categoría de los equipos enfrentados.
- El horario asignado.
- La penalización por coincidencia con otros partidos en el mismo horario.

Además, el enunciado indica que debe asignarse obligatoriamente siempre un partido el viernes y un partido el lunes.

Se resuelve el problema para **una jornada de 10 partidos**, que es el caso planteado en el enunciado.

In [24]:
# Librerías utilizadas
from itertools import product
from functools import lru_cache
import random
import pandas as pd

In [25]:
# Horarios disponibles según el enunciado.
horarios = [
    ("Viernes", 20),
    ("Sabado", 12),
    ("Sabado", 16),
    ("Sabado", 18),
    ("Sabado", 20),
    ("Domingo", 12),
    ("Domingo", 16),
    ("Domingo", 18),
    ("Domingo", 20),
    ("Lunes", 20)
]

# Audiencia base en millones para el mejor horario: sábado a las 20h.
audiencia_base = {
    ("A", "A"): 2.0,
    ("A", "B"): 1.3,
    ("A", "C"): 1.0,
    ("B", "B"): 0.9,
    ("B", "C"): 0.75,
    ("C", "C"): 0.47
}

# Coeficiente de reducción según el horario.
coef_horario = {
    ("Viernes", 20): 0.4,
    ("Sabado", 12): 0.55,
    ("Sabado", 16): 0.7,
    ("Sabado", 18): 0.8,
    ("Sabado", 20): 1.0,
    ("Domingo", 12): 0.45,
    ("Domingo", 16): 0.75,
    ("Domingo", 18): 0.85,
    ("Domingo", 20): 1.0,
    ("Lunes", 20): 0.4
}

# Penalización por coincidencia con otros partidos en el mismo horario.
penalizacion_coincidencias = {
    0: 0.00,
    1: 0.25,
    2: 0.45,
    3: 0.60,
    4: 0.70,
    5: 0.75,
    6: 0.78,
    7: 0.80,
    8: 0.80
}

# Jornada de ejemplo utilizada en el enunciado.
partidos = [
    ("Celta - Real Madrid", "B", "A"),
    ("Valencia - Real Sociedad", "B", "A"),
    ("Mallorca - Eibar", "C", "C"),
    ("Athletic - Barcelona", "B", "A"),
    ("Leganes - Osasuna", "C", "C"),
    ("Villarreal - Granada", "B", "C"),
    ("Alaves - Levante", "B", "B"),
    ("Espanyol - Sevilla", "B", "B"),
    ("Betis - Valladolid", "B", "C"),
    ("Atletico - Getafe", "B", "B")
]

In [26]:
n_partidos = len(partidos)
n_horarios = len(horarios)

posibilidades_sin_restricciones = n_horarios ** n_partidos

print("Posibilidades sin restricciones:", posibilidades_sin_restricciones)

Posibilidades sin restricciones: 10000000000


## ¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?

La restricción principal del enunciado es que debe haber al menos un partido el viernes y al menos un partido el lunes.

Para calcular las posibilidades válidas se utiliza inclusión-exclusión:

`10^10 - 2 * 9^10 + 8^10`

Donde:

- `10^10` representa todas las asignaciones posibles.
- `9^10` elimina las asignaciones sin partido el viernes.
- `9^10` elimina las asignaciones sin partido el lunes.
- `8^10` se suma porque las asignaciones sin viernes ni lunes se habían restado dos veces.

In [27]:
posibilidades_con_restricciones = (
    n_horarios**n_partidos
    - 2 * (n_horarios - 1)**n_partidos
    + (n_horarios - 2)**n_partidos
)

print("Posibilidades con restricciones:", posibilidades_con_restricciones)

Posibilidades con restricciones: 4100173022


## Modelo para el espacio de soluciones

## (*) ¿Cuál es la estructura de datos que mejor se adapta al problema? Arguméntalo.

Una solución se representa mediante una lista o tupla de 10 posiciones.

Cada posición representa un partido y el valor guardado en esa posición representa el índice del horario asignado.

Por ejemplo:

`[0, 4, 2, 8, 1, 5, 6, 7, 3, 9]`

significa que:

- El partido 0 se asigna al horario 0.
- El partido 1 se asigna al horario 4.
- El partido 2 se asigna al horario 2.
- Y así sucesivamente.

Esta estructura se adapta bien al problema porque permite evaluar fácilmente la audiencia total, contar coincidencias por horario y comprobar las restricciones de viernes y lunes.

In [28]:
# Ejemplo de asignación:
# el valor de cada posición indica el índice del horario asignado al partido.
asignacion_ejemplo = [0, 4, 2, 8, 1, 5, 6, 7, 3, 9]

for i, horario_idx in enumerate(asignacion_ejemplo):
    print(partidos[i][0], "->", horarios[horario_idx])

Celta - Real Madrid -> ('Viernes', 20)
Valencia - Real Sociedad -> ('Sabado', 20)
Mallorca - Eibar -> ('Sabado', 16)
Athletic - Barcelona -> ('Domingo', 20)
Leganes - Osasuna -> ('Sabado', 12)
Villarreal - Granada -> ('Domingo', 12)
Alaves - Levante -> ('Domingo', 16)
Espanyol - Sevilla -> ('Domingo', 18)
Betis - Valladolid -> ('Sabado', 18)
Atletico - Getafe -> ('Lunes', 20)


## Según el modelo para el espacio de soluciones

## (*) ¿Cuál es la función objetivo?

La función objetivo es maximizar la audiencia total esperada de la jornada.

Para cada partido se calcula:

`audiencia base * coeficiente del horario * factor de coincidencia`

La audiencia total es la suma de la audiencia final de todos los partidos.

## (*) ¿Es un problema de maximización o minimización?

Es un problema de **maximización**, porque se busca obtener la mayor audiencia total posible.

In [37]:
def obtener_audiencia_base(cat1, cat2):
    """
    Devuelve la audiencia base según las categorías de los equipos.

    Se ordenan las categorías para que A-B y B-A se traten como el mismo caso.
    """
    categorias = tuple(sorted((cat1, cat2)))
    return audiencia_base[categorias]


def audiencia_partido_en_horario(partido, horario):
    """
    Calcula la audiencia de un partido en un horario concreto,
    antes de aplicar penalización por coincidencias.
    """
    _, cat1, cat2 = partido

    base = obtener_audiencia_base(cat1, cat2)
    coeficiente = coef_horario[horario]

    return base * coeficiente


def obtener_penalizacion(coincidencias):
    """
    Devuelve la penalización según el número de coincidencias.

    La tabla del enunciado llega hasta 8 coincidencias. Si durante la búsqueda
    aparece un número mayor, se utiliza la mayor penalización disponible.
    """
    if coincidencias in penalizacion_coincidencias:
        return penalizacion_coincidencias[coincidencias]

    max_coincidencias = max(penalizacion_coincidencias)
    return penalizacion_coincidencias[max_coincidencias]

In [38]:
def evaluar_asignacion(asignacion, partidos, horarios):
    """
    Evalúa una asignación completa de partidos a horarios.

    Devuelve:
    - audiencia_total: audiencia total de la jornada.
    - detalle: tabla con la información de cada partido.
    """
    audiencia_total = 0
    detalle = []

    # Recorremos cada horario y buscamos qué partidos fueron asignados a él.
    for i_horario, horario in enumerate(horarios):
        indices_partidos = [
            i for i, h in enumerate(asignacion)
            if h == i_horario
        ]

        cantidad = len(indices_partidos)

        if cantidad == 0:
            continue

        # Si hay k partidos en el mismo horario, cada uno tiene k-1 coincidencias.
        coincidencias = cantidad - 1
        penalizacion = obtener_penalizacion(coincidencias)
        factor_coincidencia = 1 - penalizacion

        for i_partido in indices_partidos:
            partido = partidos[i_partido]

            base_ponderada = audiencia_partido_en_horario(partido, horario)
            audiencia_final = base_ponderada * factor_coincidencia

            audiencia_total += audiencia_final

            detalle.append({
                "Partido": partido[0],
                "Categorias": partido[1] + "-" + partido[2],
                "Horario": f"{horario[0]} {horario[1]}h",
                "Base ponderada": round(base_ponderada, 3),
                "Coincidencias": coincidencias,
                "Audiencia final": round(audiencia_final, 3)
            })

    return audiencia_total, detalle

In [39]:
audiencia_ejemplo, detalle_ejemplo = evaluar_asignacion(
    asignacion_ejemplo,
    partidos,
    horarios
)

print("Audiencia de la asignación de ejemplo:", round(audiencia_ejemplo, 3), "millones")

pd.DataFrame(detalle_ejemplo)

Audiencia de la asignación de ejemplo: 6.445 millones


,Partido,Categorias,Horario,Base ponderada,Coincidencias,Audiencia final
0,Celta - Real Madrid,B-A,Viernes 20h,0.520,0,0.520
1,Leganes - Osasuna,C-C,Sabado 12h,0.259,0,0.259
2,Mallorca - Eibar,C-C,Sabado 16h,0.329,0,0.329
3,Betis - Valladolid,B-C,Sabado 18h,0.600,0,0.600
4,Valencia - Real Sociedad,B-A,Sabado 20h,1.300,0,1.300
5,Villarreal - Granada,B-C,Domingo 12h,0.338,0,0.338
6,Alaves - Levante,B-B,Domingo 16h,0.675,0,0.675
7,Espanyol - Sevilla,B-B,Domingo 18h,0.765,0,0.765
8,Athletic - Barcelona,B-A,Domingo 20h,1.300,0,1.300
9,Atletico - Getafe,B-B,Lunes 20h,0.360,0,0.360


## Diseña un algoritmo para resolver el problema por fuerza bruta

El algoritmo por fuerza bruta genera todas las asignaciones posibles de partidos a horarios.

Para cada asignación:

1. Comprueba si hay al menos un partido el viernes y al menos un partido el lunes.
2. Calcula la audiencia total.
3. Compara el resultado con la mejor solución encontrada hasta el momento.

Este método garantiza encontrar la solución óptima, pero no es eficiente para el caso completo porque existen `10^10` asignaciones posibles.

In [40]:
def cumple_restricciones(asignacion):
    """
    Comprueba si una asignación cumple las restricciones obligatorias:
    al menos un partido en viernes y al menos un partido en lunes.
    """
    viernes = horarios.index(("Viernes", 20))
    lunes = horarios.index(("Lunes", 20))

    return viernes in asignacion and lunes in asignacion


def fuerza_bruta(partidos, horarios):
    """
    Resuelve el problema por fuerza bruta.

    Genera todas las asignaciones posibles y conserva la de mayor audiencia.
    """
    mejor_asignacion = None
    mejor_audiencia = -1

    for asignacion in product(range(len(horarios)), repeat=len(partidos)):
        if not cumple_restricciones(asignacion):
            continue

        audiencia, _ = evaluar_asignacion(asignacion, partidos, horarios)

        if audiencia > mejor_audiencia:
            mejor_audiencia = audiencia
            mejor_asignacion = asignacion

    return mejor_asignacion, mejor_audiencia

In [33]:
# Se prueba fuerza bruta con 5 partidos para evitar ejecutar 10^10 combinaciones.
partidos_pequenos = partidos[:5]

mejor_asignacion_fb, mejor_audiencia_fb = fuerza_bruta(
    partidos_pequenos,
    horarios
)

print("Mejor audiencia por fuerza bruta con 5 partidos:", round(mejor_audiencia_fb, 3))
print("Mejor asignación:", mejor_asignacion_fb)

Mejor audiencia por fuerza bruta con 5 partidos: 4.081
Mejor asignación: (4, 7, 0, 8, 9)


## Calcula la complejidad del algoritmo por fuerza bruta

Si hay `n` partidos y `h` horarios, cada partido puede asignarse a cualquiera de los `h` horarios.

Por tanto, el número de asignaciones posibles es:

`h^n`

La complejidad temporal del algoritmo por fuerza bruta es:

`O(h^n)`

En este caso concreto:

`O(10^10)`

Esto hace que la fuerza bruta no sea práctica para resolver la jornada completa.

## (*) Diseña un algoritmo que mejore la complejidad del algoritmo por fuerza bruta

Para mejorar la fuerza bruta se utiliza **programación dinámica con subconjuntos**.

La idea es procesar los horarios uno a uno y decidir qué subconjunto de partidos se asigna a cada horario.

Cada estado queda definido por:

- El horario actual.
- El conjunto de partidos pendientes de asignar.

Este enfoque evita recalcular muchas asignaciones parciales y permite resolver la jornada completa de forma exacta.

In [41]:
def audiencia_subconjunto(mask, horario_idx, partidos, horarios):
    """
    Calcula la audiencia obtenida al asignar un subconjunto de partidos
    a un horario concreto.

    El subconjunto se representa mediante una máscara de bits.
    """
    indices = [
        i for i in range(len(partidos))
        if mask & (1 << i)
    ]

    cantidad = len(indices)

    if cantidad == 0:
        return 0

    coincidencias = cantidad - 1
    penalizacion = obtener_penalizacion(coincidencias)
    factor_coincidencia = 1 - penalizacion

    horario = horarios[horario_idx]

    total = 0

    for i in indices:
        total += audiencia_partido_en_horario(partidos[i], horario)

    return total * factor_coincidencia

In [42]:
def programacion_dinamica(partidos, horarios):
    """
    Resuelve el problema mediante programación dinámica.

    En cada horario se decide qué subconjunto de partidos pendientes
    se asigna a dicho horario.
    """
    n = len(partidos)
    h = len(horarios)

    # Máscara con todos los partidos pendientes.
    todos = (1 << n) - 1

    viernes_idx = horarios.index(("Viernes", 20))
    lunes_idx = horarios.index(("Lunes", 20))

    @lru_cache(None)
    def dp(horario_idx, pendientes):
        """
        Devuelve la mejor audiencia posible desde el horario actual
        considerando los partidos pendientes.
        """
        if horario_idx == h:
            if pendientes == 0:
                return 0, []
            return -float("inf"), []

        mejor_valor = -float("inf")
        mejor_plan = None

        # Se recorren todos los subconjuntos de partidos pendientes.
        submask = pendientes

        while True:
            # Viernes y lunes deben tener al menos un partido.
            if horario_idx in [viernes_idx, lunes_idx] and submask == 0:
                pass
            else:
                valor_actual = audiencia_subconjunto(
                    submask,
                    horario_idx,
                    partidos,
                    horarios
                )

                valor_futuro, plan_futuro = dp(
                    horario_idx + 1,
                    pendientes ^ submask
                )

                valor_total = valor_actual + valor_futuro

                if valor_total > mejor_valor:
                    mejor_valor = valor_total
                    mejor_plan = [submask] + plan_futuro

            if submask == 0:
                break

            submask = (submask - 1) & pendientes

        return mejor_valor, mejor_plan

    mejor_valor, mejor_plan = dp(0, todos)

    # Convertimos el plan con máscaras a una asignación normal.
    asignacion = [None] * n

    for horario_idx, mask in enumerate(mejor_plan):
        for i in range(n):
            if mask & (1 << i):
                asignacion[i] = horario_idx

    return asignacion, mejor_valor

In [43]:
mejor_asignacion, mejor_audiencia = programacion_dinamica(
    partidos,
    horarios
)

print("Mejor audiencia encontrada:", round(mejor_audiencia, 3), "millones")
print("Mejor asignación:", mejor_asignacion)

Mejor audiencia encontrada: 6.856 millones
Mejor asignación: [8, 7, 9, 4, 0, 5, 6, 3, 1, 2]


In [44]:
audiencia_total, detalle = evaluar_asignacion(
    mejor_asignacion,
    partidos,
    horarios
)

print("Audiencia total optimizada:", round(audiencia_total, 3), "millones")

df_resultado = pd.DataFrame(detalle)
df_resultado

Audiencia total optimizada: 6.856 millones


,Partido,Categorias,Horario,Base ponderada,Coincidencias,Audiencia final
0,Leganes - Osasuna,C-C,Viernes 20h,0.188,0,0.188
1,Betis - Valladolid,B-C,Sabado 12h,0.413,0,0.413
2,Atletico - Getafe,B-B,Sabado 16h,0.630,0,0.630
3,Espanyol - Sevilla,B-B,Sabado 18h,0.720,0,0.720
4,Athletic - Barcelona,B-A,Sabado 20h,1.300,0,1.300
5,Villarreal - Granada,B-C,Domingo 12h,0.338,0,0.338
6,Alaves - Levante,B-B,Domingo 16h,0.675,0,0.675
7,Valencia - Real Sociedad,B-A,Domingo 18h,1.105,0,1.105
8,Celta - Real Madrid,B-A,Domingo 20h,1.300,0,1.300
9,Mallorca - Eibar,C-C,Lunes 20h,0.188,0,0.188


## (*) Calcula la complejidad del algoritmo

El algoritmo mejorado utiliza programación dinámica sobre subconjuntos.

Para cada horario se analizan subconjuntos de los partidos pendientes. La complejidad aproximada es:

`O(h * 3^n)`

Aunque sigue siendo exponencial, mejora frente a la fuerza bruta `O(h^n)`.

En este problema, con 10 partidos y 10 horarios, la programación dinámica permite resolver la jornada completa de forma exacta en un tiempo razonable.

## Según el problema, diseña un juego de datos de entrada aleatorio

Se genera una jornada aleatoria respetando la distribución indicada en el enunciado:

- 3 equipos de categoría A.
- 11 equipos de categoría B.
- 6 equipos de categoría C.

Después se mezclan los 20 equipos y se emparejan de dos en dos para formar 10 partidos.

In [45]:
def generar_jornada_aleatoria(seed=42):
    """
    Genera una jornada aleatoria de 10 partidos respetando la distribución:
    3 equipos A, 11 equipos B y 6 equipos C.
    """
    random.seed(seed)

    equipos = (
        [(f"A{i+1}", "A") for i in range(3)] +
        [(f"B{i+1}", "B") for i in range(11)] +
        [(f"C{i+1}", "C") for i in range(6)]
    )

    random.shuffle(equipos)

    jornada = []

    for i in range(0, len(equipos), 2):
        equipo1, cat1 = equipos[i]
        equipo2, cat2 = equipos[i + 1]

        jornada.append((
            f"{equipo1} - {equipo2}",
            cat1,
            cat2
        ))

    return jornada


partidos_aleatorios = generar_jornada_aleatoria(seed=42)

partidos_aleatorios

[('C6 - B3', 'C', 'B'),
 ('C1 - B2', 'C', 'B'),
 ('B7 - B11', 'B', 'B'),
 ('C2 - C5', 'C', 'C'),
 ('B4 - B10', 'B', 'B'),
 ('C4 - B8', 'C', 'B'),
 ('A2 - B9', 'A', 'B'),
 ('A3 - C3', 'A', 'C'),
 ('B5 - B6', 'B', 'B'),
 ('A1 - B1', 'A', 'B')]

## Aplica el algoritmo al juego de datos generado

A continuación se aplica el algoritmo mejorado de programación dinámica a la jornada aleatoria generada.

In [46]:
mejor_asignacion_random, mejor_audiencia_random = programacion_dinamica(
    partidos_aleatorios,
    horarios
)

audiencia_random, detalle_random = evaluar_asignacion(
    mejor_asignacion_random,
    partidos_aleatorios,
    horarios
)

print("Audiencia total de la jornada aleatoria:", round(audiencia_random, 3), "millones")

df_random = pd.DataFrame(detalle_random)
df_random

Audiencia total de la jornada aleatoria: 6.713 millones


,Partido,Categorias,Horario,Base ponderada,Coincidencias,Audiencia final
0,C4 - B8,C-B,Viernes 20h,0.300,0,0.300
1,C1 - B2,C-B,Sabado 12h,0.413,0,0.413
2,B5 - B6,B-B,Sabado 16h,0.630,0,0.630
3,B4 - B10,B-B,Sabado 18h,0.720,0,0.720
4,A1 - B1,A-B,Sabado 20h,1.300,0,1.300
5,C6 - B3,C-B,Domingo 12h,0.338,0,0.338
6,B7 - B11,B-B,Domingo 16h,0.675,0,0.675
7,A3 - C3,A-C,Domingo 18h,0.850,0,0.850
8,A2 - B9,A-B,Domingo 20h,1.300,0,1.300
9,C2 - C5,C-C,Lunes 20h,0.188,0,0.188


## Enumera las referencias que has utilizado

Para realizar este trabajo se han utilizado las siguientes referencias:

- Material de la asignatura Algoritmos de Optimización, VIU.
- Enunciado del Trabajo Práctico: Organización de horarios de partidos de La Liga.
- Documentación oficial de Python sobre `itertools`, `functools`, `random` y `pandas`.

## Describe brevemente las líneas de cómo crees que es posible avanzar en el estudio del problema

El problema se ha resuelto para una jornada concreta de 10 partidos, que es lo que plantea el enunciado.

Como línea futura, podría ampliarse el modelo para optimizar una temporada completa. En ese caso habría que considerar restricciones adicionales, como calendario de ida y vuelta, descansos, partidos como local y visitante, preferencias televisivas, disponibilidad de estadios, seguridad y reparto justo de horarios entre equipos.

También podrían estudiarse otras técnicas de optimización para instancias más grandes, como algoritmos genéticos, búsqueda tabú, recocido simulado o programación lineal entera.